# Inferencia manual — Modelo de recompra / churn (Olist)

Este libro **carga el modelo exportado (`.pkl`)** y permite hacer **predicciones manuales y por lote**, de forma **complementaria al job mensual automatizado en Azure Databricks** (que reentrena y registra el modelo cada día 1). Aquí **no se entrena nada**: solo se carga el pipeline ya entrenado y se predice.

> El `.pkl` contiene el pipeline completo (preprocesado + modelo): transforma datos crudos y predice sin repetir el entrenamiento. Se genera en `olist_churn_csv.ipynb` → sección **Exportación del Modelo para Producción**.

## 1. Cargar el artefacto exportado

In [1]:
import os, joblib
import pandas as pd
import numpy as np

# Usa el artefacto de producción; si no está, el del notebook de entrenamiento
MODEL_PATH = next((p for p in ["pipeline_churn_prod.pkl", "pipeline_churn_csv.pkl"]
                   if os.path.exists(p)), None)
if MODEL_PATH is None:
    raise FileNotFoundError("No se encontró el .pkl. Ejecuta la 'Exportación del Modelo "
                            "para Producción' en olist_churn_csv.ipynb y deja el .pkl junto a este libro.")

art = joblib.load(MODEL_PATH)
pipe = art["pipeline"]
FEATURES = art["features"]
meta = art.get("metadata", {})

print(f"Modelo cargado de : {MODEL_PATH}")
print(f"Tipo de modelo    : {meta.get('model_type', art.get('chosen_model', 'n/d'))}")
print(f"Nº de variables   : {len(FEATURES)}")
print(f"Variables esperadas: {FEATURES}")
art.get("target_spec", {})

Modelo cargado de : pipeline_churn_prod.pkl
Tipo de modelo    : logistic_regression
Nº de variables   : 13
Variables esperadas: ['avg_item_price', 'avg_photos_qty', 'promised_lead_days', 'max_installments', 'recency_days', 'avg_freight_value', 'n_categories', 'avg_items', 'avg_distance_km', 'n_payment_types', 'used_voucher', 'top_category', 'payment_type']


{'target': 'is_repeat',
 'horizon_days': 90,
 'poblacion': 'primerizos (frequency==1)',
 'backtest_metrics': {'AUC': 0.5731560087812477,
  'PR_AUC': 0.01728552292680392,
  'recall': 0.5111111111111111,
  'precision': 0.012497735917406267,
  'F1': 0.024398868458274398}}

## 2. Predicción manual de un cliente

Edita los valores y ejecuta. El pipeline **imputa, codifica (one-hot) y escala** automáticamente; las categorías nuevas se manejan sin error y las variables que omitas se imputan con la mediana/moda aprendidas en el entrenamiento.

In [2]:
# Edita estos valores (un cliente primerizo)
cliente = {
    "recency_days":      40,     # días desde su 1ª (y única) compra
    "avg_items":         1,      # nº de ítems del pedido
    "avg_freight_value": 18.5,   # flete (R$)
    "max_installments":  3,      # nº de cuotas
    "avg_distance_km":   350.0,  # distancia cliente-vendedor (km)
    "avg_item_price":    89.9,   # precio por ítem (R$)
    "promised_lead_days":12,     # plazo de entrega prometido (días)
    "n_categories":      1,      # categorías distintas en el pedido
    "avg_photos_qty":    3,      # fotos medias del producto
    "n_payment_types":   1,      # medios de pago distintos
    "used_voucher":      0,      # usó cupón (0/1)
    "top_category":      "bed_bath_table",  # categoría dominante
    "payment_type":      "credit_card",     # medio de pago dominante
}

X = pd.DataFrame([cliente]).reindex(columns=FEATURES)   # ordena columnas; las faltantes -> NaN (se imputan)
score = float(pipe.predict_proba(X)[0, 1])

print(f"P(recompra en 90 días) = {score:.3f}")
print(f"P(churn)               = {1 - score:.3f}")
print("Recomendación: " + ("ALTA propensión -> priorizar en campaña de conversión"
                            if score >= 0.5 else "baja propensión -> no priorizar"))

P(recompra en 90 días) = 0.513
P(churn)               = 0.487
Recomendación: ALTA propensión -> priorizar en campaña de conversión


## 3. Predicción por lote (CSV)

Carga un CSV con **una fila por cliente** y las columnas de `FEATURES` (las que falten se imputan). Devuelve el ranking por score de recompra y lo guarda en disco — listo para entregar a Marketing.

In [3]:
# En uso real:
# nuevos = pd.read_csv("clientes_nuevos.csv")
#
# Demostración con 5 clientes de ejemplo (variando recencia y ticket):
demo = pd.DataFrame([cliente] * 5)
demo["recency_days"]   = [5, 30, 90, 180, 300]
demo["avg_item_price"] = [150, 90, 60, 40, 25]

X = demo.reindex(columns=FEATURES)
demo["score_recompra"] = pipe.predict_proba(X)[:, 1]
demo["decil_prioridad"] = (pd.qcut(demo["score_recompra"].rank(method="first"),
                                   min(10, len(demo)), labels=False, duplicates="drop") + 1)

ranking = demo.sort_values("score_recompra", ascending=False)
ranking.to_csv("predicciones_clientes.csv", index=False)
print("Guardado: predicciones_clientes.csv")
ranking[["recency_days", "avg_item_price", "score_recompra", "decil_prioridad"]]

Guardado: predicciones_clientes.csv


,recency_days,avg_item_price,score_recompra,decil_prioridad
0,5,150,0.572720,5
1,30,90,0.533356,4
2,90,60,0.415649,3
3,180,40,0.254977,2
4,300,25,0.113339,1


## 4. Predicción con clientes reales (partición backtest)

A diferencia de los ejemplos anteriores (sintéticos), aquí cargamos **200 clientes reales** de la partición de **backtest** del panel (`clientes_backtest.csv`: incluye `customer_unique_id`, **todas las features no descartadas** —las 13 que usa el modelo + las 3 de control— y la etiqueta real `is_repeat`). La muestra está **enriquecida con recompradores** (60 de 200) para estimar el AUC de forma estable: el AUC es una métrica de ordenamiento e **no depende de la prevalencia**, por lo que es comparable al del backtest completo. El modelo consume solo las 13 features seleccionadas; las de control se muestran como contexto.

In [4]:
clientes = pd.read_csv("clientes_backtest.csv")
CONTROL = [c for c in ["avg_review_score", "avg_delay_days", "pct_late_deliveries"] if c in clientes.columns]

# Probabilidad de recompra (el pipeline usa solo las 13 features seleccionadas)
clientes["prob_recompra"] = pipe.predict_proba(clientes[FEATURES])[:, 1]

# Tabla: id + todas las features no descartadas + probabilidad + etiqueta real
cols_show = ["customer_unique_id"] + FEATURES + CONTROL + ["prob_recompra"]
if "is_repeat" in clientes.columns:
    cols_show += ["is_repeat"]
resultado = clientes[cols_show].sort_values("prob_recompra", ascending=False).reset_index(drop=True)

if "is_repeat" in clientes.columns and clientes["is_repeat"].nunique() > 1:
    from sklearn.metrics import roc_auc_score
    auc = roc_auc_score(clientes["is_repeat"], clientes["prob_recompra"])
    print(f"AUC en esta muestra de {len(clientes)} clientes reales: {auc:.3f} "
          "(coherente con el backtest completo, ~0,57; el AUC no depende de la prevalencia)")
print(f"Recompradores reales en la muestra: {int(clientes['is_repeat'].sum())} de {len(clientes)}")
resultado

AUC en esta muestra de 200 clientes reales: 0.550 (coherente con el backtest completo, ~0,57; el AUC no depende de la prevalencia)
Recompradores reales en la muestra: 60 de 200


,customer_unique_id,avg_item_price,avg_photos_qty,promised_lead_days,max_installments,recency_days,avg_freight_value,n_categories,avg_items,avg_distance_km,n_payment_types,used_voucher,top_category,payment_type,avg_review_score,avg_delay_days,pct_late_deliveries,prob_recompra,is_repeat
0,41b4af321ed477242aebe74652fd8219,21.93,2.0,25,8.0,17,61.24,2.0,6.0,268.65,1.0,0.0,furniture_decor,credit_card,5.0,-23.0,0.0,0.791391,1
1,d26420cca939f78a409db32872c4c8f5,35.90,1.0,9,2.0,16,36.95,1.0,5.0,125.62,1.0,0.0,bed_bath_table,credit_card,4.0,-7.0,0.0,0.701851,0
2,16a2e4356a0e4433e45df462ca7606fa,87.45,1.5,33,8.0,7,32.09,2.0,2.0,292.43,1.0,0.0,health_beauty,credit_card,4.0,-25.0,0.0,0.674551,0
3,f982853f2882760f3a52e7bdfabb1080,109.90,1.0,15,8.0,19,23.72,1.0,2.0,146.48,1.0,0.0,bed_bath_table,credit_card,5.0,-14.0,0.0,0.635996,1
4,7e094f060802ff5e25bef767446968d3,26.90,1.0,15,3.0,6,8.88,1.0,1.0,87.05,1.0,0.0,bed_bath_table,credit_card,1.0,8.0,1.0,0.608780,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,5ad755f2bc4d98be17d9f42b3c1f0c4b,69.90,10.0,38,1.0,16,19.46,1.0,1.0,1042.49,1.0,0.0,toys,credit_card,1.0,-25.0,0.0,0.336136,0
196,3db49645cbf26782b4211ccd821d9e06,499.99,7.0,15,5.0,25,53.05,1.0,1.0,112.76,1.0,0.0,cool_stuff,credit_card,5.0,-13.0,0.0,0.333267,0
197,2b3db2208861fc893211c977bf3d0244,1106.99,5.0,15,10.0,20,19.65,1.0,1.0,138.57,1.0,0.0,baby,credit_card,5.0,-14.0,0.0,0.317367,0
198,f44ffa618df5f8a65899275c93b14ae2,1260.00,6.0,30,10.0,19,118.64,1.0,2.0,1249.55,1.0,0.0,computers_accessories,credit_card,4.0,-21.0,0.0,0.272295,0


## 5. Relación con la automatización en Azure

- **Automático (Azure Databricks):** el job `olist-job-entrenamiento-churn` **reentrena** el modelo y lo registra cada día 1 (MLflow + Model Registry); la inferencia mensual rankea **toda la base**. Detalle en `olist_churn_csv.ipynb` §18–19 y en el diagrama *Arquitectura del pipeline mensual*.
- **Manual (este libro):** carga el `.pkl` exportado y predice **clientes puntuales o lotes ad-hoc sin reentrenar** — útil para análisis rápidos, validaciones puntuales o integraciones externas que no esperan al ciclo mensual.

Ambos caminos usan **exactamente el mismo artefacto** (`.pkl`), por lo que las predicciones manuales son consistentes con las automáticas.